In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from src.data import load_data
from src.config import RANDOM_STATE, CV_DIR
from src.feature_groups import group_columns
from src.experiment_log import ExperimentLogger

# Tải dữ liệu train và khởi tạo logger cho nhóm thí nghiệm D
X_train, _, y_train, _, labels = load_data()
lg = ExperimentLogger("Khanh")


def rf():
    # Khởi tạo mô hình Random Forest với cấu hình đã chọn
    clf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        max_features="log2",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    return clf


def kiem_dinh_cap(ten_goc, ten_khac):
    # Đọc điểm macro_f1 qua từng fold của mô hình gốc và biến thể bị bỏ nhóm
    df_goc = pd.read_csv(CV_DIR / f"{ten_goc}.csv")
    df_khac = pd.read_csv(CV_DIR / f"{ten_khac}.csv")

    a = df_goc["macro_f1"].to_numpy()
    b = df_khac["macro_f1"].to_numpy()

    d = b - a
    k = len(d)
    chenh_tb = d.mean()

    # Kiểm định t hiệu chỉnh Nadeau-Bengio cho các fold CV không hoàn toàn độc lập
    var_d = d.var(ddof=1)
    sai_so_chuan = np.sqrt((1.0 / k + 0.25) * var_d)
    t_stat = chenh_tb / sai_so_chuan

    # Kiểm định 2 phía với bậc tự do df = k - 1
    p_val = 2 * (1.0 - stats.t.cdf(abs(t_stat), df=k - 1))

    return round(chenh_tb, 4), round(t_stat, 2), round(p_val, 3)


nhom_dac_trung = group_columns(X_train.columns)
rows = []

for g, cols in nhom_dac_trung.items():
    ten = "rf_bo_nhom_" + g
    X_sub = X_train.drop(columns=cols)
    model_rf = rf()
    ghi_chu = "bo nhom " + g + " gom " + str(len(cols)) + " cot"

    # Chạy đánh giá và ghi log cho biến thể bỏ nhóm đặc trưng g
    lg.run(
        model_rf,
        X_sub,
        y_train,
        ten,
        strategy="N0 khong xu ly",
        notes=ghi_chu,
    )

    # So sánh với bản đầy đủ (random_forest_tuned)
    ch, t_val, p_val = kiem_dinh_cap("random_forest_tuned", ten)
    cv = pd.read_csv(CV_DIR / f"{ten}.csv")

    thong_tin = {
        "nhom_bi_bo": g,
        "so_cot": len(cols),
        "macro_f1": round(cv["macro_f1"].mean(), 4),
        "chenh_so_voi_day_du": ch,
        "t": t_val,
        "p": p_val,
        "recall_c4": round(cv["recall_c4"].mean(), 3),
        "recall_c5": round(cv["recall_c5"].mean(), 3),
        "recall_c11": round(cv["recall_c11"].mean(), 3),
    }
    rows.append(thong_tin)

# Sắp xếp tăng dần theo chenh_so_voi_day_du để nhóm làm giảm điểm mạnh nhất nằm trên cùng
bang = pd.DataFrame(rows)
bang = bang.sort_values("chenh_so_voi_day_du")
bang.to_csv(CV_DIR / "ablation_groups_rf.csv", index=False)

bang

,nhom_bi_bo,so_cot,macro_f1,chenh_so_voi_day_du,t,p,recall_c4,recall_c5,recall_c11
3,co_TCP,9,0.9555,-0.0075,-1.06,0.347,0.760,0.82,0.891
0,cong,2,0.9568,-0.0062,-3.70,0.021,0.760,0.82,0.897
4,payload,11,0.9591,-0.0039,-0.74,0.501,0.693,0.82,0.880
7,cua_so_TCP,3,0.9607,-0.0023,-0.61,0.578,0.727,0.82,0.874
1,so_goi_va_luong,13,0.9618,-0.0012,-0.37,0.733,0.727,0.82,0.909
8,proto_service,13,0.9621,-0.0009,-0.57,0.602,0.760,0.82,0.897
5,thoi_gian_giua_goi_IAT,10,0.9627,-0.0003,-0.28,0.797,0.760,0.82,0.891
2,kich_thuoc_header,4,0.9637,0.0007,0.68,0.535,0.760,0.82,0.903
6,active_idle,5,0.9649,0.0019,1.68,0.168,0.760,0.82,0.914
